In [11]:
import torch
from transformers import AutoTokenizer
from vul_detector import VulDetector

MAX_LEN = 256
# -------------------------
# 6) Inference function
# -------------------------
def predict_vulnerability(
    code_text: str,
    model_path: str = None,
    model_instance=None,
    tokenizer_instance=None,
    device_instance=None,
    return_probabilities: bool = False
):
    """Run a single vulnerability prediction."""
    model_to_use = model_instance if model_instance is not None else model
    tokenizer_to_use = tokenizer_instance if tokenizer_instance is not None else tokenizer
    device_to_use = device_instance if device_instance is not None else device
    
    if model_path is not None:
        checkpoint = torch.load(model_path, map_location=device_to_use)
        model_to_use.load_state_dict(checkpoint)
        model_to_use.to(device_to_use)
        print(f"Loaded checkpoint: {model_path}")
    
    inputs = tokenizer_to_use(
        code_text.strip(),
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
        add_special_tokens=True,
        return_tensors="pt"
)
    inputs = {k: v.to(device_to_use) for k, v in inputs.items()}
    
    model_to_use.eval()
    with torch.no_grad():
        outputs = model_to_use(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        predicted_class = torch.argmax(probs, dim=-1).item()
    
    if return_probabilities:
        return {
            "safe": probs[0][0].item(),
            "vulnerable": probs[0][1].item(),
            "predicted_class": predicted_class
        }
    return "vulnerable" if predicted_class == 1 else "safe"

In [ ]:
from pathlib import Path

model_name = "microsoft/unixcoder-base"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = VulDetector(model_name=model_name, num_labels=2)

checkpoint_path = Path("model/best_model_epoch_4.pt")
if checkpoint_path.is_file():
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict)
    print(f"Loaded checkpoint: {checkpoint_path}")
else:
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

model.to(device)
model.eval()

sample_code = """
void unsafe_copy(char *dest, char *src) {
        strcpy(dest, src);  // No bounds checking
    }
"""
code_text = "Analyze this C function for vulnerability:\n" + sample_code

result = predict_vulnerability(
    code_text,
    model_instance=model,
    tokenizer_instance=tokenizer,
    device_instance=device,
    return_probabilities=True
)
print("\nSingle prediction:", result)





Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/unixcoder-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded checkpoint: model/best_model_epoch_4.pt

Single prediction: {'safe': 0.8940560817718506, 'vulnerable': 0.10594398528337479, 'predicted_class': 0}
